# 03 · Modelado XGBoost — Predicción de éxito de videojuegos

## Contexto
Entrenamiento del modelo de clasificación para predecir si un videojuego 
será un éxito, usando el dataset preparado en el notebook anterior.

## Objetivo
- Entrenar modelo XGBoost con control de desbalanceo de clases
- Evaluar con métricas apropiadas para clases desbalanceadas (ROC-AUC, F1, PR-AUC)
- Ajustar umbral óptimo de clasificación
- Guardar modelo y features para uso en FastAPI

## Decisiones técnicas
- **Exclusión de leakage features**: game_rating, ratings_count, game_added y otras 
  variables derivadas del target se excluyen para evitar data leakage
- **scale_pos_weight**: compensa el desbalanceo ~75/25 entre negativos y positivos
- **Umbral ajustable**: se optimiza F1 en lugar de usar 0.5 por defecto

## Input
`data/processed/train_dataset.parquet`

## Output
`models/xgb_success_model_[timestamp].pkl` · `models/success_features.pkl`

# Modelado (XGBoost) — Predicción de éxito

Este notebook:
- Carga el dataset model-ready (`02_feature_engineering`).
- Preprocesa: One-Hot para categóricas.
- Entrena XGBoost con control de desbalanceo.
- Evalúa y guarda el modelo.


## 0) Setup


In [ ]:
# !pip install -U pandas numpy scikit-learn xgboost joblib matplotlib

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix, precision_recall_curve
)

import matplotlib.pyplot as plt
from xgboost import XGBClassifier

RANDOM_STATE = 42


## 1) Cargar dataset


In [ ]:
data_path = Path("../../data/processed/train_dataset.parquet")
if not data_path.exists():
    raise FileNotFoundError("No encuentro data/processed/train_dataset.parquet. Ejecuta primero feature_engineering.")

df = pd.read_parquet(data_path)
df.shape, df.head(3)


In [ ]:
df.columns



## 2) Definir X / y


In [ ]:
"""
Para evitar proxy leakage se excluyen variables de popularidad directa:
- owned
- toplay
- suggestions_count
- game_added
- playing
- rating_popularity_ratio
- engagement_score
- quality_confidence

"""


In [ ]:
 ALL_NUMERIC_FEATURES = [
    'game_rating', 'ratings_count',
    'game_added', 'suggestions_count',
    'playtime', 'playing', 'owned', 'toplay',
    'num_platforms', 'num_stores', 'num_genres', 'num_tags',
    'rating_popularity_ratio', 'engagement_score',
    'years_since_release', 'quality_confidence',
    'recency_score'
]

LEAKAGE_FEATURES = [
    'game_rating','ratings_count','game_added','playtime','playing',
    'rating_popularity_ratio','quality_confidence','engagement_score',
    'owned','toplay','suggestions_count'
]

NUMERIC_FEATURES = [
    f for f in ALL_NUMERIC_FEATURES
    if f not in LEAKAGE_FEATURES
]

CATEGORICAL_FEATURES = [
    'esrb_name', 'release_year',
    'has_multiplayer', 'has_singleplayer', 'is_indie'
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

TARGET = "is_success_v2"

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

print(y.value_counts(normalize=True))

print("X columns:", X.columns.tolist())
print("suggestions_count en X?", "suggestions_count" in X.columns)


## 3) Split train/test


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("suggestions_count en X_train?", "suggestions_count" in X_train.columns)
print(X_train.shape, X_test.shape)



## 4) Preprocesado


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ("num", "passthrough", NUMERIC_FEATURES),
    ],
    remainder="drop",
)


## 5) Entrenamiento XGBoost


In [ ]:
pos = int(y_train.sum())
neg = int((y_train == 0).sum())
scale_pos_weight = (neg / max(pos, 1))

scale_pos_weight


In [ ]:
xgb = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("xgb", xgb),
])

model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

roc = roc_auc_score(y_test, y_proba)
pr = average_precision_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)

roc, pr, f1


## 6) Reporte + matriz de confusión


In [ ]:
print(classification_report(y_test, y_pred, digits=4))
confusion_matrix(y_test, y_pred)


## 7) Ajuste de umbral (mejor F1)


In [ ]:
prec, rec, thr = precision_recall_curve(y_test, y_proba)
f1s = (2 * prec[:-1] * rec[:-1]) / np.clip((prec[:-1] + rec[:-1]), 1e-9, None)

best_idx = int(np.argmax(f1s))
best_thr = float(thr[best_idx])
best_f1 = float(f1s[best_idx])

best_thr, best_f1


In [ ]:
y_pred_best = (y_proba >= best_thr).astype(int)
print("Threshold:", best_thr)
print(classification_report(y_test, y_pred_best, digits=4))
confusion_matrix(y_test, y_pred_best)


## 8) Curva Precision-Recall


In [ ]:
plt.figure(figsize=(7,5))
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve")
plt.grid(True)
plt.show()


In [ ]:
# Mostrar los features importantes para el XGBoost 

preprocess = model.named_steps["preprocess"]

feature_names = preprocess.get_feature_names_out()

importances = model.named_steps["xgb"].feature_importances_

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
)

importance_df.head(15)


## 9) Guardar modelo


In [ ]:
import joblib
from datetime import datetime
from pathlib import Path

out_dir = Path("models")
out_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = out_dir / f"xgb_success_model_{stamp}.pkl"
success_features = out_dir/ f"success_features.pkl"

joblib.dump(model, model_path)
joblib.dump(FEATURES, success_features) 
model_path
success_features


In [ ]:
# Comprobación
from pathlib import Path
import joblib

models_dir = Path("models")

# Ver modelos disponibles
model_files = list(models_dir.glob('xgb_success_model_*.pkl'))
print(f"Modelos encontrados: {len(model_files)}")
for f in model_files:
    print(f"  - {f.name}")

# Verificar features
features_file = models_dir / 'success_features.pkl'
if features_file.exists():
    features = joblib.load(features_file)
    print(f"\nFeatures file OK: {len(features)} features")
else:
    print(f"\nFeatures file NO encontrado: {features_file}")

In [ ]:
import joblib

features = joblib.load('models/success_features.pkl')
print("Orden correcto de features:")
for i, feat in enumerate(features):
    print(f"{i}: {feat}")

In [ ]:
# En el notebook
print(df['game_rating'].dtype)
print(df['game_rating'].describe())
print(df['game_rating'].head(10))